In [ ]:
# BLOCK 1: helpers + prompt loader + one turn runner
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

BASE_URL = "http://127.0.0.1:8500"
PROMPT_SEARCH_DIRS = [Path("."), Path("prompts")]
ROLES = ["G", "H"]

def http_json(method, path, payload=None, timeout=300):
    url = f"{BASE_URL}{path}"
    data = None
    headers = {}
    if payload is not None:
        data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

def role_snapshot(role):
    return http_json("GET", f"/api/admin/role/{urllib.parse.quote(role)}")

def send_command(role, action, payload=None):
    return http_json("POST", "/api/admin/command", {
        "role": role,
        "action": action,
        "payload": payload or {},
    })["command"]

def wait_command(command_id, timeout=300, print_every=1.0):
    started = time.time()
    last_print = 0
    while True:
        data = http_json("GET", f"/api/admin/command/{command_id}")
        now = time.time()
        if now - last_print >= print_every:
            print(f"[{time.strftime('%H:%M:%S')}] cmd={command_id[:8]} status={data['status']} done={data['done']}")
            last_print = now
        if data["done"]:
            return data["result"]
        if timeout and now - started > timeout:
            raise TimeoutError(f"Timeout waiting for command_id={command_id}")
        time.sleep(0.2)

def run_command(role, action, payload=None, timeout=300, print_every=1.0):
    cmd = send_command(role, action, payload)
    print(f"{action} -> role={role}, command_id={cmd['command_id']}")
    return wait_command(cmd["command_id"], timeout=timeout, print_every=print_every)

def load_prompt(role):
    for base in PROMPT_SEARCH_DIRS:
        path = base / f"{role}.txt"
        if path.exists():
            return path.read_text(encoding="utf-8").strip()
    raise FileNotFoundError(f"Khong tim thay prompt cho role {role} (G.txt/H.txt)")

PROMPTS = {role: load_prompt(role) for role in ROLES}
PERSONA_LOADED = {role: False for role in ROLES}

def wait_role_ready(role, attempts=20, sleep_s=1):
    for i in range(attempts):
        snap = role_snapshot(role)
        print(f"[{role}] check {i}: status={snap['status']} sessions={snap['sessions']}")
        if snap["status"] != "OFFLINE" or snap.get("dom_info"):
            return snap
        time.sleep(sleep_s)
    raise RuntimeError(f"Role {role} chua online")

def compose_message(role, task_text):
    if PERSONA_LOADED[role]:
        return task_text
    PERSONA_LOADED[role] = True
    return f"{PROMPTS[role]}\n\n{'=' * 40}\n{task_text}"

def run_role_turn(role, text, timeout=1800):
    message = compose_message(role, text)
    result = {}
    result["probe"] = run_command(role, "PROBE", timeout=60, print_every=2)
    result["set_prompt"] = run_command(role, "SET_PROMPT", {
        "text": message,
        "method": "auto",
        "samples": 6,
        "sample_ms": 300,
    }, timeout=120, print_every=2)
    result["find_send"] = run_command(role, "FIND_SEND", timeout=60, print_every=2)
    result["click_send"] = run_command(role, "CLICK_SEND", timeout=60, print_every=2)
    result["assistant"] = run_command(role, "WAIT_ASSISTANT_DONE", timeout=timeout, print_every=2)
    result["sync"] = run_command(role, "SYNC_TRANSCRIPT", {
        "reason": f"loop_{role.lower()}_post_turn_sync"
    }, timeout=60, print_every=2)
    result["text"] = (result["assistant"].get("text") or "").strip()
    return result

for role in ROLES:
    wait_role_ready(role)

print("Ready:", ROLES)

In [ ]:
# BLOCK 2: G/H loop until H says TASK COMPLETE
INITIAL_TASK = """[ROLE]
You are G, the Developer in a closed MCP workflow.

[WORKSPACE]
Use MCP: linux-mcp
ROOT_PATH: /home/ayumi/Workspace/git_project/qmh
"""

MAX_ROUNDS = 12
gh_log = []

current_for_g = INITIAL_TASK

for round_no in range(1, MAX_ROUNDS + 1):
    print("\n" + "=" * 100)
    print(f"ROUND {round_no} - G")
    print("=" * 100)
    g_turn = run_role_turn("G", current_for_g, timeout=1800)
    g_text = g_turn["text"]
    print(g_text[:2000])

    gh_log.append({
        "round": round_no,
        "role": "G",
        "prompt": current_for_g,
        "response": g_text,
    })

    review_prompt = f"""Day la bao cao moi nhat tu G:

{g_text}

Hay review/test theo prompt cua ban.
Neu da dat goal thi chi in dung:
TASK COMPLETE

Neu chua dat, hay noi ro G phai sua/them gi tiep theo.
"""

    print("\n" + "=" * 100)
    print(f"ROUND {round_no} - H")
    print("=" * 100)
    h_turn = run_role_turn("H", review_prompt, timeout=1800)
    h_text = h_turn["text"]
    print(h_text[:2000])

    gh_log.append({
        "round": round_no,
        "role": "H",
        "prompt": review_prompt,
        "response": h_text,
    })

    if "TASK COMPLETE" in h_text:
        print("\nTASK COMPLETE")
        break

    current_for_g = f"""Day la nhan xet/test moi nhat tu H:

{h_text}

Hay sua tiep theo dung cac diem H yeu cau.
Sau khi sua xong, bao cao lai bang chung cu the.
"""
else:
    print(f"\nDung do dat MAX_ROUNDS={MAX_ROUNDS} ma H chua bao TASK COMPLETE")

Path("gh_loop_log.json").write_text(
    json.dumps(gh_log, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("Saved: gh_loop_log.json")